Process addresses data

- ingest data into lakehouse.
- perform data quality checks
- apply changes and write to csd type 2

In [0]:
import dlt
from pyspark.sql import functions as F

create bronze table

In [0]:
@dlt.table(
name = "circuitbox.bronze.addresses",
comment = "Raw files with ingestion timestamp",
table_properties = {'quality':'bronze'}
)

def create_bronze_addresses():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "true")
        .option(
            "cloudFiles.schemaLocation",
            "/Volumes/circuitbox/landing/operational_data/_checkpoints/address_schema/",
        )
        .load("/Volumes/circuitbox/landing/operational_data/address/")
        .select(
            "*",
            F.col("_metadata.file_path").alias("input_file_path"),
            F.current_timestamp().alias("current_timestamp"),
        )
    )

create silver table


In [0]:
@dlt.table(
name = "circuitbox.lakehouse.addresses",
comment = "Implemented data quality checks and changed datatype of columns",
table_properties = {'quality':'staging'}
)

@dlt.expect_or_fail("valid_customer_id","customer_id IS NOT NULL")
@dlt.expect_or_drop("valid_address","address_line_1 IS NOT NULL")
@dlt.expect("valid_postcode","len(POSTCODE)=5")

def create_staging_addresses_clean():
    return(
        spark.readStream.table("LIVE.circuitbox.bronze.addresses")
        .select("customer_id"
                ,"address_line_1"
                ,"city"
                ,"state"
                ,"postcode"
                ,F.col("created_date").cast("date")
                )
        
    )

CREATE SILVER TABLE

In [0]:
dlt.create_streaming_table(name = "circuitbox.silver.addresses"
           ,comment = "address data stored as SCD TYPE 2"
           ,table_properties= {'quality':'silver'}
           )

dlt.apply_changes(
  source = "circuitbox.lakehouse.addresses",
  target = "circuitbox.silver.addresses",
  keys = ["customer_id"],
  sequence_by = "created_date",
  stored_as_scd_type = 2  
  )